# Collaborative Filtering — KNN Based Recommender

This notebook uses the **Surprise** library to implement user-based KNN collaborative filtering on course enrollment data.

## 1. Setup

In [18]:
!pip install scikit-surprise==1.1.1 -q
import pandas as pd
import numpy as np
from surprise import Dataset, Reader, KNNBasic
from surprise.model_selection import cross_validate, train_test_split
from surprise import accuracy

ratings_df = pd.read_csv('https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-ML321EN-SkillsNetwork/labs/datasets/ratings.csv')
print('Dataset:', ratings_df.shape)
print('Rating values:', ratings_df['rating'].unique())

  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> [53 lines of output]
      <string>:45: _DeprecatedInstaller: setuptools.installer and fetch_build_eggs are deprecated.
      !!
      
              ********************************************************************************
              Requirements should be satisfied by a PEP 517 installer.
              If you are using pip, you can try `pip install --use-pep517`.
      
              This deprecation is overdue, please update your project and remove deprecated
              calls to avoid build errors in the future.
              ********************************************************************************
      
      !!
      C:\Users\KIIT0001\OneDrive\Desktop\Programs\ML_DL_NLP_Bootcamp\venv\python.exe: No module named pip
      Traceback (most recent call last):
        File "C:\Users\KIIT0001\AppData\Local\Temp\pip-build-env-joaqp1us\over

Dataset: (233306, 3)
Rating values: [3. 2.]


## 2. Build Surprise Dataset

In [19]:
reader = Reader(rating_scale=(2, 3))
data = Dataset.load_from_df(ratings_df[['user','item','rating']], reader)
trainset, testset = train_test_split(data, test_size=0.2, random_state=42)
print(f'Training set: {trainset.n_users} users, {trainset.n_items} items')
print(f'Test set size: {len(testset)} ratings')

Training set: 32253 users, 126 items
Test set size: 46662 ratings


## 3. KNN with MSD Similarity

In [22]:
sim_options_msd = {'name': 'msd', 'user_based': True}
knn_msd = KNNBasic(k=40, sim_options=sim_options_msd)
knn_msd.fit(trainset)
predictions_msd = knn_msd.test(testset)
rmse_msd = accuracy.rmse(predictions_msd)
print(f'KNN (MSD): RMSE = {rmse_msd:.4f}')

Computing the msd similarity matrix...
Done computing similarity matrix.
RMSE: 0.1905
KNN (MSD): RMSE = 0.1905


## 4. KNN with Cosine Similarity

In [55]:
sim_options_cos = {'name': 'cosine', 'user_based': True}
knn_cos = KNNBasic(k=40, sim_options=sim_options_cos)
knn_cos.fit(trainset)
predictions_cos = knn_cos.test(testset)
rmse_cos = accuracy.rmse(predictions_cos)
print(f'KNN (Cosine): RMSE = {rmse_cos:.4f}')

Computing the cosine similarity matrix...


MemoryError: Unable to allocate 7.75 GiB for an array with shape (32253, 32253) and data type float64

## 5. Compare Both Approaches

In [56]:
import matplotlib.pyplot as plt

models = ['KNN (MSD)', 'KNN (Cosine)']
rmses = [rmse_msd, rmse_cos]

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(models, rmses, color=['#E74C3C', '#2ECC71'], width=0.4, edgecolor='white')
for bar, val in zip(bars, rmses):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{val:.4f}', ha='center', va='bottom', fontweight='bold')
ax.set_ylabel('RMSE (lower is better)')
ax.set_title('KNN Collaborative Filtering — RMSE Comparison')
ax.set_ylim(0, 1.2)
ax.axhline(0.2063, color='green', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.savefig('knn_comparison.png', dpi=120, bbox_inches='tight')
plt.show()

NameError: name 'rmse_cos' is not defined

## Summary

| Model | Similarity | RMSE |
|---|---|---|
| KNN User-Based | MSD | 0.1905 |
| KNN User-Based | Cosine | Not available (MemoryError on similarity matrix) |

**Key insight:** KNN user-based collaborative filtering with MSD similarity achieves RMSE = 0.1905 on this dataset. The cosine similarity variant could not be evaluated due to memory constraints (7.75 GB required for a 32,253 × 32,253 similarity matrix) — this remains future work, e.g. by subsampling users or switching to item-based similarity (126 × 126 matrix instead).